MongoDB connection

In [1]:
from pymongo import MongoClient

In [2]:
client = MongoClient("mongodb://localhost:27017/")

In [3]:
db = client["ecommerce_recommendation"]

In [4]:
reviews_collection = db["reviews"]
interactions_collection = db["interactions"]

Top 5 highest-rated products

In [5]:
pipeline = [
    {
        "$group": {

            "_id": "$asin",

            "average_rating": {
                "$avg": "$overall"
            },

            "total_reviews": {
                "$sum": 1
            }
        }
    },
    {
        "$sort": {
            "average_rating": -1
        }
    },

    {
        "$limit": 5
    }
]

In [6]:
result = reviews_collection.aggregate(pipeline)

In [7]:
for item in result:
    print(item)

{'_id': 'B00004TENT', 'average_rating': 5.0, 'total_reviews': 14}
{'_id': '3930992868', 'average_rating': 5.0, 'total_reviews': 7}
{'_id': 'B000067V8L', 'average_rating': 5.0, 'total_reviews': 6}
{'_id': 'B000067RMY', 'average_rating': 5.0, 'total_reviews': 5}
{'_id': 'B000063VY6', 'average_rating': 5.0, 'total_reviews': 5}


Average rating by sentiment

In [8]:
pipeline = [
    {
        "$group": {

            "_id": "$analytics.sentiment",

            "average_rating": {
                "$avg": "$overall"
            },

            "review_count": {
                "$sum": 1
            }
        }
    }
]

In [9]:
result = reviews_collection.aggregate(pipeline)

In [10]:
for item in result:
    print(item)

{'_id': 'neutral', 'average_rating': 3.0, 'review_count': 3177}
{'_id': 'positive', 'average_rating': 4.7541363941632575, 'review_count': 32758}
{'_id': 'negative', 'average_rating': 1.4217619166458697, 'review_count': 4007}


Most active users

In [11]:
pipeline = [
    {
        "$group": {

            "_id": "$user_id",

            "total_interactions": {
                "$sum": 1
            }
        }
    },

    {
        "$sort": {
            "total_interactions": -1
        }
    },

    {
        "$limit": 10
    }
]

In [12]:
result = interactions_collection.aggregate(pipeline)

In [13]:
for item in result:
    print(item)

{'_id': 'A231WM2Z2JL0U3', 'total_interactions': 109}
{'_id': 'A5JLAU2ARJ0BO', 'total_interactions': 44}
{'_id': 'AT2J7H5TRZM8Z', 'total_interactions': 30}
{'_id': 'A1MJMYLRTZ76ZX', 'total_interactions': 28}
{'_id': 'A2VV0TJNJT9D3O', 'total_interactions': 23}
{'_id': 'A6FIAB28IS79', 'total_interactions': 22}
{'_id': 'A1RPTVW5VEOSI', 'total_interactions': 21}
{'_id': 'A2BGZ52M908MJY', 'total_interactions': 21}
{'_id': 'A2R6RA8FRBS608', 'total_interactions': 20}
{'_id': 'A1NVD0TKNS1GT5', 'total_interactions': 20}


Positive Reviwes only

In [14]:
pipeline = [
    {
        "$match": {
            "analytics.sentiment": "positive"
        }
    },
    {
        "$group": {

            "_id": "$asin",

            "positive_reviews": {
                "$sum": 1
            }
        }
    },
    {
        "$sort": {
            "positive_reviews": -1
        }
    },
    {
        "$limit": 5
    }
]

In [15]:
result = reviews_collection.aggregate(pipeline)

In [16]:
for item in result:
    print(item)

{'_id': 'B00004ZCJE', 'positive_reviews': 1018}
{'_id': 'B00005T3G0', 'positive_reviews': 651}
{'_id': 'B00004T8R2', 'positive_reviews': 463}
{'_id': 'B000067RT6', 'positive_reviews': 453}
{'_id': 'B00001P4ZH', 'positive_reviews': 423}


Unwinding helpful votes

In [17]:
pipeline = [
    {
        "$unwind": "$helpful"
    },
    {
        "$group": {

            "_id": "$helpful",

            "count": {
                "$sum": 1
            }
        }
    },
    {
        "$sort": {
            "count": -1
        }
    }
]

In [18]:
result = reviews_collection.aggregate(pipeline)

In [19]:
for item in result:
    print(item)

{'_id': 0, 'count': 40274}
{'_id': 1, 'count': 12510}
{'_id': 2, 'count': 5932}
{'_id': 3, 'count': 3817}
{'_id': 4, 'count': 2707}
{'_id': 5, 'count': 1899}
{'_id': 6, 'count': 1442}
{'_id': 7, 'count': 1161}
{'_id': 8, 'count': 943}
{'_id': 9, 'count': 718}
{'_id': 10, 'count': 706}
{'_id': 11, 'count': 576}
{'_id': 12, 'count': 487}
{'_id': 13, 'count': 483}
{'_id': 14, 'count': 393}
{'_id': 15, 'count': 353}
{'_id': 17, 'count': 300}
{'_id': 16, 'count': 294}
{'_id': 18, 'count': 262}
{'_id': 19, 'count': 224}
{'_id': 20, 'count': 222}
{'_id': 21, 'count': 192}
{'_id': 23, 'count': 187}
{'_id': 22, 'count': 174}
{'_id': 25, 'count': 154}
{'_id': 24, 'count': 146}
{'_id': 26, 'count': 134}
{'_id': 27, 'count': 131}
{'_id': 30, 'count': 112}
{'_id': 29, 'count': 108}
{'_id': 35, 'count': 94}
{'_id': 28, 'count': 93}
{'_id': 31, 'count': 90}
{'_id': 34, 'count': 90}
{'_id': 33, 'count': 86}
{'_id': 32, 'count': 83}
{'_id': 36, 'count': 74}
{'_id': 39, 'count': 71}
{'_id': 38, 'count':

Customer Engagement Analysis

In [20]:
pipeline = [
    {
        "$project": {

            "_id": 0,

            "product_id": "$asin",

            "rating": "$overall",

            "sentiment": "$analytics.sentiment",

            "review_length": "$analytics.review_length",

            "helpful_ratio": "$analytics.helpful_ratio",

            "engagement_score": {

                "$multiply": [
                    "$analytics.helpful_ratio",
                    "$analytics.review_length"
                ]
            }
        }
    },
    {
        "$sort": {
            "engagement_score": -1
        }
    },
    {
        "$limit": 10
    }
]

In [21]:
result = reviews_collection.aggregate(pipeline)

In [22]:
for item in result:
    print(item)

{'product_id': 'B00004SY4H', 'rating': 5, 'sentiment': 'positive', 'review_length': 2744, 'helpful_ratio': 0.8965517241000001, 'engagement_score': 2460.1379309304}
{'product_id': '1400532655', 'rating': 4, 'sentiment': 'positive', 'review_length': 2079, 'helpful_ratio': 1.0, 'engagement_score': 2079.0}
{'product_id': 'B000058TLP', 'rating': 5, 'sentiment': 'positive', 'review_length': 1664, 'helpful_ratio': 0.9900990099, 'engagement_score': 1647.5247524736}
{'product_id': '1400501466', 'rating': 1, 'sentiment': 'negative', 'review_length': 1799, 'helpful_ratio': 0.9122807018, 'engagement_score': 1641.1929825381999}
{'product_id': 'B00005LEOH', 'rating': 5, 'sentiment': 'positive', 'review_length': 1564, 'helpful_ratio': 0.976744186, 'engagement_score': 1527.627906904}
{'product_id': 'B00005LE76', 'rating': 5, 'sentiment': 'positive', 'review_length': 1418, 'helpful_ratio': 1.0, 'engagement_score': 1418.0}
{'product_id': 'B000065BPB', 'rating': 4, 'sentiment': 'positive', 'review_length